necessary libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

loading necessary files

In [2]:
BASE = Path("/Users/muskaanchugh/epidemics-art-forecasting/synthetic-data-generation")
DATASETS = BASE / "datasets"
FINAL = BASE / "final_datasets"
FINAL.mkdir(exist_ok=True)

In [3]:
synthetic_dfs = {
    f.stem.replace("_pre_calibration", ""): pd.read_csv(f)
    for f in DATASETS.glob("*.csv")
}

defining monthly dosage configuration

In [4]:
dosage_config_adults = {
    'abc/3tc - adult': 30,
    'azt/3tc - adult': 60,
    'atv - adult': 30,
    'drv - adult': 60
}

In [5]:
dosage_config_pediatric = {
    'abc/3tc - pediatric - pre 2022': 138,
    'abc/3tc - pediatric - post 2022': 69,
    'azt/3tc - pediatric': 138,
    'lpv 100/25 - pediatric': 111
}

In [6]:
dosage_config_booster = {
    'rtv - booster': 30
}

defining the calibration function

In [7]:
def calibrate(df, dose_per_month):
    df = df.copy()
    df["consumption"] = df["patients"] * dose_per_month
    return df[["month", "year", "consumption"]]

calibrating adult regimens

In [8]:
adult_key_map = {
    'abc/3tc - adult': 'abc_3tc_adult',
    'azt/3tc - adult': 'azt_3tc_adult',
    'atv - adult':     'atv_adult',
    'drv - adult':     'drv_adult'
}

calibrated_adults = {}
for config_key, df_key in adult_key_map.items():
    dose = dosage_config_adults[config_key]
    calibrated_adults[df_key] = calibrate(synthetic_dfs[df_key], dose)

calibrating booster

In [9]:
calibrated_booster = {
    'rtv_booster': calibrate(synthetic_dfs["rtv_booster"], dosage_config_booster["rtv - booster"])
}

calibrating pediatric regimens

In [10]:
month_map = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

calibrated_pediatric = {}

# abc+3tc pediatric — split at april 2022
abc_ped = synthetic_dfs["abc_3tc_pediatric"].copy()
abc_ped["month_num"] = abc_ped["month"].str.lower().map(month_map)
abc_ped["t"] = abc_ped["year"] * 12 + abc_ped["month_num"]
split_t = 2022 * 12 + 4

pre  = abc_ped[abc_ped["t"] < split_t].copy()
post = abc_ped[abc_ped["t"] >= split_t].copy()

pre["consumption"]  = pre["patients"]  * dosage_config_pediatric["abc/3tc - pediatric - pre 2022"]
post["consumption"] = post["patients"] * dosage_config_pediatric["abc/3tc - pediatric - post 2022"]

calibrated_pediatric["abc_3tc_pediatric"] = pd.concat([pre, post])[["month", "year", "consumption"]].reset_index(drop=True)

# azt+3tc and lpv pediatric — standard
pediatric_key_map = {
    'azt/3tc - pediatric':    'azt_3tc_pediatric',
    'lpv 100/25 - pediatric': 'lpv_pediatric'
}

for config_key, df_key in pediatric_key_map.items():
    dose = dosage_config_pediatric[config_key]
    calibrated_pediatric[df_key] = calibrate(synthetic_dfs[df_key], dose)

converting to MM-YYYY index and saving

In [11]:
MONTH_NUM = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

def to_date_index(df):
    df = df.copy()
    m = df["month"].str.lower().map(MONTH_NUM)
    df["date"] = pd.to_datetime(
        df["year"].astype(str) + "-" + m.apply(lambda x: f"{x:02d}") + "-01"
    )
    return df[["date", "consumption"]].set_index("date")

for name, df in {**calibrated_adults, **calibrated_pediatric, **calibrated_booster}.items():
    converted = to_date_index(df)
    converted.to_csv(FINAL / f"{name}_synthetic.csv")